# 직접 해보기 04 · 표범 무늬의 원소 세기

NASA PDS가 공개한 퍼서비어런스 PIXL 자료(sol 1188, 체야바 폭포, doi:10.17189/vth5-0676)로 반점의 테두리·속이 바탕보다 얼마나 진한지 센다.

같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
ox, roi = A.load()
print({k: len(v) for k, v in roi.items()})

## 1. 측정점 지도를 그린다
`A.build_points()`가 만든 `data/points.csv`의 좌표를 쓴다. matplotlib이 없으면 `pip install matplotlib`.

In [ ]:
import csv
import matplotlib.pyplot as plt
A.build_points()
pts = list(csv.DictReader(open('data/points.csv', encoding='utf-8')))
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for ax, col in zip(axes, ['P2O5', 'SO3']):
    s = ax.scatter([float(p['u_mm']) for p in pts], [float(p['v_mm']) for p in pts], c=[float(p[col]) for p in pts], s=6, cmap='Reds')
    cores = [p for p in pts if p['group'] == 'cores']
    ax.scatter([float(p['u_mm']) for p in cores], [float(p['v_mm']) for p in cores], s=22, facecolors='none', edgecolors='k', lw=.6)
    ax.set_title(col + ' (wt%) · 검은 네모 = 반점 속'); ax.set_aspect('equal'); fig.colorbar(s, ax=ax)
axes[1].set_xlabel('띠 방향 (mm)'); plt.tight_layout()

## 2. 세 무리의 평균을 내고 바탕으로 나눈다 (책의 예상 결과)

In [ ]:
for region in ('rims', 'cores'):
    for oxide in ('P2O5', 'SO3', 'FeO-T', 'CaO', 'NiO'):
        e = A.enrichment(ox, roi, region, oxide)
        print(f"{A.REGIONS[region]:<8} {oxide:<6} {e['mean']:.2f}% ÷ {e['base']:.2f}% = {e['ratio']:.2f}배 (±{e['err']:.2f})")

## 3. 함정을 점검한다
띠 전체 1,649점의 SO₃·CaO 평균은 바탕과 왜 이렇게 다른가? 오른쪽 끝의 흰 맥을 빼면 어떻게 되는지 `u_mm` 좌표로 직접 나눠 보자.

In [ ]:
for oxide in ('SO3', 'CaO'):
    m, se, n = A.mean_se(ox, roi['band'], oxide)
    print(oxide, f'띠 전체 {m:.2f}% ({n:,}점)')